# 🎯 Hull Tactical Market Prediction - Inference Server

## ⚠️ IMPORTANT: This notebook MUST run on Kaggle platform!

**This is an INFERENCE COMPETITION notebook.**
- ❌ Cannot run locally (kaggle_evaluation module not available)
- ✅ Must upload and run on Kaggle notebook environment
- ✅ Requires Kaggle's inference competition infrastructure

## 📌 Setup Instructions

### Before Running This Notebook:

1. **Upload Dataset to Kaggle:**
   - Go to: https://www.kaggle.com/datasets
   - Upload `src/` and `conf/` folders
   - Name it: `prediction-market-modules`

2. **Add Dataset to This Notebook:**
   - In Kaggle Notebook: "Add Data" → "Your Datasets" → `prediction-market-modules`

3. **Run on Kaggle:**
   - This notebook uses `kaggle_evaluation` module
   - Only available in Kaggle's inference competition environment
   - Local execution will show warnings but won't break

## 🚀 This Notebook:
- ✅ Uses Kaggle Evaluation API (streaming predictions)
- ✅ Returns single float per timestep
- ✅ Loads model once (first prediction only)
- ✅ 5-minute response time per prediction
- ✅ 15-minute server startup time limit

## 1️⃣ Setup Module Paths

In [ ]:
import os
import sys
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print("="*80)
print("SETTING UP MODULE PATHS")
print("="*80)

# Add kaggle_evaluation to path (from competition data)
kaggle_eval_path = Path("/kaggle/input/hull-tactical-market-prediction")
if kaggle_eval_path.exists():
    sys.path.insert(0, str(kaggle_eval_path))
    print(f"✓ Added kaggle_evaluation path: {kaggle_eval_path}")

# Add custom modules to path
dataset_dir = Path("/kaggle/input/prediction-market-modules")
if dataset_dir.exists():
    sys.path.insert(0, str(dataset_dir))
    print(f"✓ Added to path: {dataset_dir}")
else:
    print(f"⚠️  Dataset not found: {dataset_dir}")
    print(f"Available datasets:")
    for d in Path("/kaggle/input/").iterdir():
        print(f"  - {d.name}")

print("\n✅ Path setup complete!")

## 2️⃣ Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb

# Try to import kaggle_evaluation (only available in Kaggle environment)
try:
    import kaggle_evaluation.default_inference_server
    print("✓ kaggle_evaluation imported")
    KAGGLE_ENV = True
except ImportError:
    print("⚠️  kaggle_evaluation not found - this is normal if running locally")
    print("   This notebook MUST be run on Kaggle platform for submission")
    KAGGLE_ENV = False

print("="*80)
print("IMPORTING MODULES")
print("="*80)

try:
    from src.data import DataLoader
    print("✓ DataLoader imported")
except Exception as e:
    print(f"❌ DataLoader import failed: {e}")

try:
    from src.features import FeatureEngineering
    print("✓ FeatureEngineering imported")
except Exception as e:
    print(f"❌ FeatureEngineering import failed: {e}")

try:
    from src.models import ReturnPredictor
    print("✓ ReturnPredictor imported")
except Exception as e:
    print(f"❌ ReturnPredictor import failed: {e}")

try:
    from src.utils import load_config, Timer
    print("✓ Utils imported")
except Exception as e:
    print(f"❌ Utils import failed: {e}")

print("\n✅ All imports successful!")

if not KAGGLE_ENV:
    print("\n" + "="*80)
    print("⚠️  WARNING: Not running in Kaggle environment!")
    print("="*80)
    print("This notebook requires Kaggle's inference competition environment.")
    print("Please upload and run this notebook on Kaggle platform.")
    print("="*80)

## 3️⃣ Load and Train Model (Once)

This cell trains the model and stores it in global variables.
The model will be loaded once and reused for all predictions.

In [ ]:
print("="*80)
print("LOADING CONFIGURATION AND TRAINING MODEL")
print("="*80)

# Configuration
config_path = "/kaggle/input/prediction-market-modules/conf/params.yaml"
config = load_config(config_path)
print(f"✓ Configuration loaded")

# Load training data
train_path = "/kaggle/input/hull-tactical-market-prediction/train.csv"
data_loader = DataLoader(config_path=config_path)
train_df, _ = data_loader.load_data(train_path, train_path)  # Only need train data
print(f"✓ Training data loaded: {train_df.shape}")

# Feature engineering
fe = FeatureEngineering(config_path=config_path)
with Timer("Feature engineering"):
    train_features = fe.fit_transform(train_df)
print(f"✓ Features engineered: {train_features.shape}")

# Get feature columns for prediction (SAVE GLOBALLY for predict function)
FEATURE_COLS = [
    col for col in train_features.columns 
    if col not in ['date_id', 'forward_returns', 'risk_free_rate', 'market_forward_excess_returns']
]
print(f"✓ Feature columns saved: {len(FEATURE_COLS)} features")

# Train model
predictor = ReturnPredictor(model_type='lightgbm', config_path=config_path)
with Timer("Model training"):
    results = predictor.train_cv(
        df=train_features,
        target_col='forward_returns',
        date_col='date_id'
    )

print(f"\n✅ Model trained successfully!")
print(f"CV Score: {results['mean_score']:.6f} ± {results['std_score']:.6f}")
print(f"\n⚡ Model ready for inference!")

## 4️⃣ Define Prediction Function

This function will be called for each timestep.
It receives a Polars DataFrame and must return a single float value.

In [ ]:
def predict(test: pl.DataFrame) -> float:
    """
    Predict function called by Kaggle Evaluation API.
    
    Args:
        test: Polars DataFrame with test data for current timestep
        
    Returns:
        float: Single prediction value
    """
    # Convert Polars to Pandas (our pipeline uses Pandas)
    test_df = test.to_pandas()
    
    # Feature engineering
    test_features = fe.transform(test_df)
    
    # Use EXACT same features as training (CRITICAL!)
    # This prevents feature count mismatch errors
    X = test_features[FEATURE_COLS]
    
    # Make prediction
    prediction = predictor.predict(X)
    
    # Return single float value
    # If multiple rows, return mean (or you can use first value)
    if len(prediction) == 1:
        return float(prediction[0])
    else:
        # If batch contains multiple rows, average them
        return float(np.mean(prediction))

print("✅ predict() function defined")
print("\nFunction signature: predict(test: pl.DataFrame) -> float")
print("  - Input: Polars DataFrame with test data")
print("  - Output: Single float prediction value")
print(f"  - Features used: {len(FEATURE_COLS)} (same as training)")

## 5️⃣ Initialize Inference Server

In [ ]:
print("="*80)
print("INITIALIZING INFERENCE SERVER")
print("="*80)

if not KAGGLE_ENV:
    print("\n⚠️  Skipping server initialization (not in Kaggle environment)")
    print("Please run this notebook on Kaggle platform")
else:
    # Create inference server with our predict function
    inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
    
    print("\n✅ Inference server initialized!")
    print("\nServer configuration:")
    print("  - Prediction function: predict()")
    print("  - Response time limit: 5 minutes per prediction")
    print("  - Server startup limit: 15 minutes")
    print("\n⚡ Ready to serve predictions!")

## 6️⃣ Start Server (Kaggle Evaluation Mode)

**This cell starts the actual server when running on Kaggle's evaluation system.**

- In competition mode: Waits for timestep-by-timestep data from Kaggle
- In local mode: Runs local gateway for testing

In [ ]:
print("="*80)
print("STARTING INFERENCE SERVER")
print("="*80)

if not KAGGLE_ENV:
    print("\n⚠️  Cannot start server (not in Kaggle environment)")
    print("This notebook must be run on Kaggle platform for submission")
else:
    if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        # Running in actual Kaggle evaluation
        print("\n🚀 Running in COMPETITION MODE")
        print("Waiting for evaluation API requests...")
        inference_server.serve()
    else:
        # Running locally or in notebook mode
        print("\n🧪 Running in LOCAL TEST MODE")
        print("Using local gateway for testing...")
        inference_server.run_local_gateway((
            '/kaggle/input/hull-tactical-market-prediction/',
        ))
    
    print("\n✅ Server execution complete!")

## 📊 Summary

### What This Notebook Does:

1. **Loads custom modules** from uploaded dataset
2. **Trains model once** using all training data
3. **Defines predict()** function that:
   - Receives Polars DataFrame (timestep data)
   - Applies feature engineering
   - Returns single float prediction
4. **Starts inference server** that:
   - Waits for Kaggle evaluation API requests
   - Calls predict() for each timestep
   - Returns predictions within 5-minute limit

### Key Differences from Batch Submission:

| Aspect | Batch Mode | Inference Server Mode |
|--------|------------|----------------------|
| Data | All at once | Timestep by timestep |
| Prediction | DataFrame | Single float |
| Output | Parquet file | API response |
| Timing | No limit | 5 min per prediction |

### 🎯 Ready for Submission!

Just run all cells and the server will handle Kaggle's evaluation requests automatically.